# Market Basket Model

## Learning Objectives

1. **Define** items, baskets, support, confidence, and interest
2. **Compute** frequent itemsets and association rules from a small database
3. **Explain** the support threshold trade-off
4. **Identify** applications beyond retail

## The Model

**Items:** a finite set $I = \{i_1, i_2, \ldots, i_n\}$ (e.g., grocery products)

**Baskets:** a collection of transactions $B = \{b_1, b_2, \ldots, b_m\}$ where each $b_k \subseteq I$

**Support of an itemset $X$:**
$$\text{supp}(X) = \frac{|\{b \in B : X \subseteq b\}|}{|B|}$$

The fraction of baskets containing all items in $X$.

**Frequent itemset:** $X$ with $\text{supp}(X) \geq s$ for threshold $s \in (0,1)$

## Association Rules

An **association rule** $X \to Y$ (where $X \cap Y = \emptyset$) has:

**Confidence:**
$$\text{conf}(X \to Y) = \frac{\text{supp}(X \cup Y)}{\text{supp}(X)}$$
Probability of $Y$ given basket contains $X$.

**Interest (lift):**
$$\text{interest}(X \to Y) = \text{conf}(X \to Y) - \text{supp}(Y)$$
How much more likely $Y$ is given $X$, compared to baseline.

- Interest > 0: positive association
- Interest < 0: negative association (Y less likely given X)
- Interest ≈ 0: X and Y are independent

In [1]:
from collections import defaultdict
from itertools import combinations

def count_support(baskets, itemsets):
    """Count support for a list of itemsets."""
    counts = defaultdict(int)
    for basket in baskets:
        basket_set = frozenset(basket)
        for itemset in itemsets:
            if frozenset(itemset) <= basket_set:
                counts[frozenset(itemset)] += 1
    n = len(baskets)
    return {k: v/n for k,v in counts.items()}

# Small transaction database
baskets = [
    ['bread','milk','butter'],
    ['bread','milk'],
    ['milk','butter','eggs'],
    ['bread','butter'],
    ['bread','milk','eggs'],
    ['milk','eggs'],
    ['bread','milk','butter','eggs'],
    ['butter','eggs'],
]
items = sorted(set(i for b in baskets for i in b))
print(f"Items: {items}")
print(f"Baskets: {len(baskets)}")

# Compute support for all 1-itemsets and 2-itemsets
singles = [[i] for i in items]
pairs   = list(combinations(items, 2))
s1 = count_support(baskets, singles)
s2 = count_support(baskets, pairs)

print("""
Item support:""")
for k,v in sorted(s1.items(), key=lambda x:-x[1]):
    bar='#'*int(v*20)
    print(f"  {set(k)}: {v:.3f}  {bar}")

Items: ['bread', 'butter', 'eggs', 'milk']
Baskets: 8

Item support:
  {'milk'}: 0.750  ###############
  {'bread'}: 0.625  ############
  {'butter'}: 0.625  ############
  {'eggs'}: 0.625  ############


In [2]:
# Find frequent itemsets at threshold s=0.3
s = 0.3
freq_1 = {k:v for k,v in s1.items() if v >= s}
freq_2 = {k:v for k,v in s2.items() if v >= s}

print(f"Frequent 1-itemsets (s≥{s}):")
for k,v in sorted(freq_1.items(), key=lambda x:-x[1]):
    print(f"  {set(k)}: {v:.3f}")

print(f"""
Frequent 2-itemsets (s≥{s}):""")
for k,v in sorted(freq_2.items(), key=lambda x:-x[1]):
    print(f"  {set(k)}: {v:.3f}")

# Compute association rules from frequent 2-itemsets
print(f"""
Association rules (conf ≥ 0.5):""")
all_supp = {**s1, **s2}
for pair, supp_pair in freq_2.items():
    items_pair = list(pair)
    for lhs in [frozenset([items_pair[0]]), frozenset([items_pair[1]])]:
        rhs = pair - lhs
        supp_lhs = all_supp.get(lhs, 0)
        conf = supp_pair / supp_lhs if supp_lhs > 0 else 0
        interest = conf - all_supp.get(rhs, 0)
        if conf >= 0.5:
            print(f"  {set(lhs)} → {set(rhs)}: conf={conf:.3f}, interest={interest:+.3f}")

Frequent 1-itemsets (s≥0.3):
  {'milk'}: 0.750
  {'bread'}: 0.625
  {'butter'}: 0.625
  {'eggs'}: 0.625

Frequent 2-itemsets (s≥0.3):
  {'milk', 'bread'}: 0.500
  {'eggs', 'milk'}: 0.500
  {'butter', 'bread'}: 0.375
  {'butter', 'milk'}: 0.375
  {'butter', 'eggs'}: 0.375

Association rules (conf ≥ 0.5):
  {'butter'} → {'bread'}: conf=0.600, interest=-0.025
  {'bread'} → {'butter'}: conf=0.600, interest=-0.025
  {'milk'} → {'bread'}: conf=0.667, interest=+0.042
  {'bread'} → {'milk'}: conf=0.800, interest=+0.050
  {'butter'} → {'milk'}: conf=0.600, interest=-0.150
  {'milk'} → {'butter'}: conf=0.500, interest=-0.125
  {'butter'} → {'eggs'}: conf=0.600, interest=-0.025
  {'eggs'} → {'butter'}: conf=0.600, interest=-0.025
  {'eggs'} → {'milk'}: conf=0.800, interest=+0.050
  {'milk'} → {'eggs'}: conf=0.667, interest=+0.042


## The Support Threshold Trade-off

| Low threshold $s$ | High threshold $s$ |
|-------------------|-------------------|
| Many frequent itemsets | Few frequent itemsets |
| Includes rare patterns | Only common patterns |
| High computational cost | Low computational cost |
| May include noise | Misses interesting rare rules |

**Typical values:** $s = 0.01$ to $s = 0.1$ for retail (millions of transactions).
Even 1% support = tens of thousands of baskets.

## Applications Beyond Retail

| Domain | Items | Baskets | Association Rule Example |
|--------|-------|---------|--------------------------|
| Web mining | Web pages | User sessions | `{sports.com} → {espn.com}` |
| Medical | Symptoms/diagnoses | Patient records | `{fever, cough} → {flu}` |
| Bioinformatics | Genes | Experiments (gene sets) | `{BRCA1} → {BRCA2}` |
| NLP | Words | Documents | `{deep, learning} → {neural}` |
| Clickstream | Products viewed | Sessions | `{phone case} → {screen protector}` |

The key insight: anywhere there are "sets of co-occurring items" in "transactions" of data.